# 05 — The Noether context and the atom tree

Substitute for the discharged context-tree / holographic-memory notebooks.
`H(t) = (S(t), H(t-1), D, R, N)`; `S(t)` is a lattice element with declared
generators; the follow matrix is tropical; and the sparse Merkle tree over
`B` is the sketch-space presentation — `HLLSet(LUT) = HLLSet(MerkleTree)`.


In [2]:
:dep hllset-contracts = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/hllset-next-v2/crates/hllset-contracts" }
:dep hllset-core = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/hllset-next-v2/crates/hllset-core" }
:dep hllset-lut = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/hllset-next-v2/crates/hllset-lut" }
:dep hllset-context = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/hllset-next-v2/crates/hllset-context" }


In [3]:
use hllset_contracts::token_in_bytes;
use hllset_context::{evolve, invariants_hold, Context, FollowMatrix};
use hllset_core::HLLSet;
use hllset_lut::{AtomTree, LutNode};
fn hll(ids: &[u32]) -> HLLSet { HLLSet::from_tokens(ids.iter().map(|&n| token_in_bytes(n))) }
println!("Noether context + atom tree loaded");


Noether context + atom tree loaded


---
## S(t) with declared generators

The state is the join of its declared leaves; insert/remove are persistent.


In [4]:
let ga = hll(&[0, 1, 2]);
let gb = hll(&[2, 3]);
let ctx: Context = Context::from_generators(vec![ga.clone(), gb.clone()]);
println!("S(t) = join of {} generators, popcount {}",
    ctx.generators().len(), ctx.state().popcount());
let gc = hll(&[9]);
let with_c = ctx.with_generator(gc.clone());
println!("insert then remove -> back to {} generators: {}",
    with_c.without_generator(&gc).generators().len(),
    with_c.without_generator(&gc).generators().len() == ctx.generators().len());


S(t) = join of 2 generators, popcount 4
insert then remove -> back to 2 generators: true


---
## Evolution: D, R, N

D = departed, R = retained, N = novel — with the three invariants.


In [5]:
let a = hll(&[0, 1, 2]);
let b = hll(&[3, 4]);
let c = hll(&[2, 5, 6]);
let previous: HLLSet = HLLSet::union_all(vec![a.clone(), b.clone()]);
let current: HLLSet = HLLSet::union_all(vec![a.clone(), c.clone()]);
let n = evolve(&previous, &current);
println!("D={} R={} N={}  invariants={}",
    n.departed.popcount(), n.retained.popcount(), n.novel.popcount(),
    invariants_hold(&n, &previous, &current));


D=2 R=3 N=2  invariants=true


---
## The tropical follow matrix

`⊕ = max`, `⊗ = +`: grow-only, merge is a join, restriction is a projection.


In [6]:
let mut fm: FollowMatrix = FollowMatrix::default();
fm.observe(b"a", b"b", 2);
fm.observe(b"a", b"b", 5);
fm.observe(b"b", b"c", 3);
println!("follow(a,b) = {} (max), follow(b,c) = {}", fm.get(b"a", b"b"), fm.get(b"b", b"c"));
let active: std::collections::BTreeSet<Vec<u8>> =
    [b"a".to_vec(), b"b".to_vec()].into_iter().collect();
let proj = fm.restrict(&active);
println!("restrict to {{a,b}}: follow(a,b)={} follow(b,c)={} (projected away)",
    proj.get(b"a", b"b"), proj.get(b"b", b"c"));


follow(a,b) = 5 (max), follow(b,c) = 3
restrict to {a,b}: follow(a,b)=5 follow(b,c)=0 (projected away)


---
## The atom tree is the same element

Leaves are the active atoms `{i}`, the root is the Merkle root — and
ingesting the LUT equals joining the tree.


In [7]:
let lut: LutNode = LutNode::new("tokenLUT_all", (0..300u32).map(token_in_bytes));
let from_tokens: HLLSet = lut.to_hllset();
let tree: AtomTree = AtomTree::from_lut(&lut);
let tree_atoms: Vec<u32> = from_tokens.bit_addresses().iter().map(|a| a.bit()).collect();
println!("ingest(LUT) atoms: {} ; MerkleTree atoms: {}", tree_atoms.len(), tree.atoms.len());
println!("same atoms: {}", tree_atoms == tree.atoms);
println!("root is a 40-hex SHA-1: {}", tree.root().len() == 40);


ingest(LUT) atoms: 288 ; MerkleTree atoms: 288
same atoms: true
root is a 40-hex SHA-1: true


---
## The finest granularity: K_i restores to the atom

Each leaf pairs the atom with its fiber — the tree is the restoration rule


In [8]:
let i = hllset_contracts::BitAddress::of_token(&token_in_bytes(44)).bit();
let k_i: LutNode = LutNode::fiber("K_i", i, (0..64u32).map(token_in_bytes));
println!("K_{} has {} tokens; ingest(K_i) popcount {} (the atom)",
    i, k_i.tokens.len(), k_i.to_hllset().popcount());


K_13728 has 1 tokens; ingest(K_i) popcount 1 (the atom)
